# HRL OPC UA Methodentests mit Username/Password

Dieses Notebook testet die am `HRL_SkillSet` exponierten OPC-UA-Methoden.

Wichtige Trennung:
- `username` und `password` sind fuer die Identifizierung am OPC-UA-Server.
- `SKILLSET_NAME` ist nur der Browse-Name des PLC-Objekts, unter dem die HRL-Skills gefunden werden.

Das Notebook unterstuetzt beides:
- reine Anmeldung per `username`/`password`
- optional zusaetzlich `Basic256Sha256` mit Client-Zertifikat

Wenn dein UAExpert-Test mit `username`/`password` schon funktioniert hat, kannst du `USE_SECURITY = False` lassen und zuerst damit testen.

In [139]:
from __future__ import annotations

from contextlib import contextmanager
from getpass import getpass
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Tuple
import threading
import time

import pyads
from opcua import Client, ua

PROJECT_DIR = Path.cwd()
CERT_DIR = PROJECT_DIR / "certs"

CLIENT_CERT_PEM = CERT_DIR / "client_cert.pem"
CLIENT_CERT_DER = CERT_DIR / "client_cert.der"
CLIENT_KEY_PEM = CERT_DIR / "client_key.pem"

APPLICATION_URI = "urn:ma:implementierung-sim-ma:python-opcua-client"
OPC_ENDPOINT = "opc.tcp://DESKTOP-LNJR8E0:4840"
PLC_ROOT_NAME = "PLC1"  # UAExpert: Root/Objects/PLC1/...
SKILLSET_NAME = "HRL_SkillSet"
SKILLSET_NODEID = "ns=4;s=HRL_SkillSet"
AUSLAGERN_METHOD_NODEID = "ns=4;s=HRL_SkillSet.HRL_NMethod_Auslagern.HRL_NMethod_Auslagern"
HRL_AUSLAGERN_SKILL_NAME = "HRL_NMethod_Auslagern"
OPC_REQUEST_TIMEOUT_SECONDS = 45.0
OPC_SECURE_CHANNEL_TIMEOUT_MS = 300000
OPC_SESSION_TIMEOUT_MS = 300000

USE_SECURITY = True
SECURITY_POLICY = "Basic256Sha256"
SECURITY_MODE = "SignAndEncrypt"

AMS_NET_ID = "10.1.47.44.1.1"
ADS_PORT = 851

CHECKSTATE_POLL = 0.2
ABORT_DELAY = 0.2

HRL_SKILLS = [
    "HRL_Skill_CB_MoveForwards",
    "HRL_Skill_CB_MoveBackwards",
    "HRL_Skill_RGB_MoveBackwards",
    "HRL_Skill_RGB_MoveForwards",
    "HRL_Skill_RGB_MoveVerticalForwards",
    "HRL_Skill_RGB_MoveVerticalBackwards",
    "HRL_Skill_RGB_MoveHorizontalForwards",
    "HRL_Skill_RGB_MoveHorizontalBackwards",
]

ADS_BOOL_SYMBOLS = {
    "notaus_a": "GVL_Sim.diNotAus_Kanal_A",
    "notaus_b": "GVL_Sim.diNotAus_Kanal_B",
    "hrl_reset_button": "HRL_SkillSet.ResetButton",
    "hrl_start_button": "HRL_SkillSet.StartButton",
    "hrl_confirmation_button": "HRL_SkillSet.ConfirmationButton",
    "vsg_reset_button": "VSG_SkillSet.ResetButton",
    "vsg_start_button": "VSG_SkillSet.StartButton",
    "vsg_confirmation_button": "VSG_SkillSet.ConfirmationButton",
    "ts_horizontal": "GVL_HRL_Sim.HRL_Ref_Taster_horizontal",
    "ts_vertical": "GVL_HRL_Sim.HRL_Ref_Taster_vertikal",
    "ts_ausleger_vorne": "GVL_HRL_Sim.HRL_Ref_Taster_Ausleger_vorne",
    "ts_ausleger_hinten": "GVL_HRL_Sim.HRL_Ref_Taster_Ausleger_hinten",
    "ls_inner": "GVL_HRL_Sim.HRL_LS_innen",
    "ls_outer": "GVL_HRL_Sim.HRL_LS_aussen",
    "sim": "GVL_Sim.bSimMode"
}

ENCODER_SYMBOLS = {
    "horizontal_1": "GVL_HRL_Sim.HRL_Enc_horizontal_I1",
    "horizontal_2": "GVL_HRL_Sim.HRL_Enc_horizontal_I2",
    "vertical_1": "GVL_HRL_Sim.HRL_Enc_vertikal_I1",
    "vertical_2": "GVL_HRL_Sim.HRL_Enc_vertikal_I2",
}

In [129]:
OPC_USERNAME = "AdminVD"
OPC_PASSWORD = "123456"

#if not OPC_USERNAME:
 #   raise ValueError("Bitte einen OPC-UA-Username eingeben.")

print({
    "endpoint": OPC_ENDPOINT,
    "username": OPC_USERNAME,
    "password": OPC_PASSWORD,
    "use_security": USE_SECURITY,
    "skillset_name": SKILLSET_NAME,
})

{'endpoint': 'opc.tcp://DESKTOP-LNJR8E0:4840', 'username': 'AdminVD', 'password': '123456', 'use_security': True, 'skillset_name': 'HRL_SkillSet'}


In [130]:
security_string = (
    f"{SECURITY_POLICY},{SECURITY_MODE},{CLIENT_CERT_DER},{CLIENT_KEY_PEM}"
)

if USE_SECURITY:
    missing = [path for path in [CLIENT_CERT_PEM, CLIENT_CERT_DER, CLIENT_KEY_PEM] if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Fehlende Zertifikatsdateien: {missing}")

print("Projektordner:", PROJECT_DIR)
print("Cert-Ordner:", CERT_DIR)
print("Application URI:", APPLICATION_URI)
print("Security aktiviert:", USE_SECURITY)
if USE_SECURITY:
    print("Security String:", security_string)

Projektordner: f:\MA\Implementierung_Sim_MA
Cert-Ordner: f:\MA\Implementierung_Sim_MA\certs
Application URI: urn:ma:implementierung-sim-ma:python-opcua-client
Security aktiviert: True
Security String: Basic256Sha256,SignAndEncrypt,f:\MA\Implementierung_Sim_MA\certs\client_cert.der,f:\MA\Implementierung_Sim_MA\certs\client_key.pem


In [131]:
@contextmanager
def open_ads():
    plc = pyads.Connection(AMS_NET_ID, ADS_PORT)
    plc.open()
    try:
        yield plc
    finally:
        plc.close()


def prepare_plc_for_skill_tests() -> Dict[str, bool]:
    values = {
        "sim": True,
        "notaus_a": True,
        "notaus_b": True,
        "hrl_reset_button": True,
        "hrl_start_button": True,
        "hrl_confirmation_button": False,
        "vsg_reset_button": True,
        "vsg_start_button": True,
        "vsg_confirmation_button": False,
        "ts_horizontal": False,
        "ts_vertical": False,
        "ts_ausleger_vorne": False,
        "ts_ausleger_hinten": True,
        "ls_inner": True,
        "ls_outer": False,
    }
    with open_ads() as plc:
        for key, value in values.items():
            plc.write_by_name(ADS_BOOL_SYMBOLS[key], value, pyads.PLCTYPE_BOOL)
    return values


def read_encoder_snapshot() -> Dict[str, int]:
    with open_ads() as plc:
        return {
            name: int(plc.read_by_name(symbol, pyads.PLCTYPE_UDINT))
            for name, symbol in ENCODER_SYMBOLS.items()
        }


def resolve_ads_udint_symbol(symbol_or_key: str) -> str:
    return ENCODER_SYMBOLS.get(symbol_or_key, symbol_or_key)


def write_ads_udint(symbol_or_key: str, value: int) -> Dict[str, Any]:
    symbol = resolve_ads_udint_symbol(symbol_or_key)
    with open_ads() as plc:
        plc.write_by_name(symbol, int(value), pyads.PLCTYPE_UDINT)
    return {
        "symbol": symbol,
        "value": int(value),
    }


def schedule_ads_udint_write(symbol_or_key: str, value: int, delay_seconds: float) -> Dict[str, Any]:
    symbol = resolve_ads_udint_symbol(symbol_or_key)

    def worker() -> None:
        try:
            time.sleep(delay_seconds)
            with open_ads() as plc:
                plc.write_by_name(symbol, int(value), pyads.PLCTYPE_UDINT)
        except Exception as exc:
            print({
                "scheduled_ads_write_failed": True,
                "symbol": symbol,
                "error": repr(exc),
            })

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    return {
        "symbol": symbol,
        "value": int(value),
        "delay_seconds": float(delay_seconds),
        "scheduled": True,
    }


def schedule_ads_udint_sequence(events: List[Tuple[float, str, int]]) -> List[Dict[str, Any]]:
    scheduled: List[Dict[str, Any]] = []
    for delay_seconds, symbol_or_key, value in events:
        if delay_seconds <= 0:
            scheduled.append(write_ads_udint(symbol_or_key, value))
        else:
            scheduled.append(schedule_ads_udint_write(symbol_or_key, value, delay_seconds))
    return scheduled


def read_ads_bool(symbol_or_key: str) -> Dict[str, Any]:
    symbol = ADS_BOOL_SYMBOLS.get(symbol_or_key, symbol_or_key)
    with open_ads() as plc:
        value = bool(plc.read_by_name(symbol, pyads.PLCTYPE_BOOL))
    return {
        "symbol": symbol,
        "value": value,
    }


def read_hrl_signal_diagnostic() -> Dict[str, Dict[str, Any]]:
    return {
        "ls_inner_sim": read_ads_bool("ls_inner"),
        "ls_inner_runtime": read_ads_bool("GVL_HRL.HRL_LS_innen"),
        "ls_outer_sim": read_ads_bool("ls_outer"),
        "ls_outer_runtime": read_ads_bool("GVL_HRL.HRL_LS_aussen"),
    }


def resolve_ads_bool_symbol(symbol_or_key: str) -> str:
    return ADS_BOOL_SYMBOLS.get(symbol_or_key, symbol_or_key)


def write_ads_bool(symbol_or_key: str, value: bool) -> Dict[str, Any]:
    symbol = resolve_ads_bool_symbol(symbol_or_key)
    with open_ads() as plc:
        plc.write_by_name(symbol, value, pyads.PLCTYPE_BOOL)
    return {
        "symbol": symbol,
        "value": bool(value),
    }


def write_ads_bool_after_delay(symbol_or_key: str, value: bool, delay_seconds: float) -> Dict[str, Any]:
    symbol = resolve_ads_bool_symbol(symbol_or_key)
    time.sleep(delay_seconds)
    with open_ads() as plc:
        plc.write_by_name(symbol, value, pyads.PLCTYPE_BOOL)
    return {
        "symbol": symbol,
        "value": bool(value),
        "delay_seconds": float(delay_seconds),
    }


def schedule_ads_bool_write(symbol_or_key: str, value: bool, delay_seconds: float) -> Dict[str, Any]:
    symbol = resolve_ads_bool_symbol(symbol_or_key)

    def worker() -> None:
        try:
            time.sleep(delay_seconds)
            with open_ads() as plc:
                plc.write_by_name(symbol, value, pyads.PLCTYPE_BOOL)
        except Exception as exc:
            print({
                "scheduled_ads_write_failed": True,
                "symbol": symbol,
                "error": repr(exc),
            })

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    return {
        "symbol": symbol,
        "value": bool(value),
        "delay_seconds": float(delay_seconds),
        "scheduled": True,
    }


def schedule_ads_bool_sequence(events: List[Tuple[float, str, bool]]) -> List[Dict[str, Any]]:
    scheduled: List[Dict[str, Any]] = []
    for delay_seconds, symbol_or_key, value in events:
        if delay_seconds <= 0:
            scheduled.append(write_ads_bool(symbol_or_key, value))
        else:
            scheduled.append(schedule_ads_bool_write(symbol_or_key, value, delay_seconds))
    return scheduled


def schedule_hrl_auslagern_ausleger_sequence(step30_start_seconds: float = 4.4) -> List[Dict[str, Any]]:
    # Die Ausleger-Taster duerfen erst umschalten, wenn der Skill wirklich in Schritt 30 ist.
    # Mit der aktuellen Encoder-Sequenz beginnt Schritt 30 live nach ca. 4.2 s.
    # Initialzustand (hinten=True, vorne=False) wird bereits in prepare_plc_for_skill_tests() gesetzt.
    return schedule_ads_bool_sequence([
        (step30_start_seconds, "ts_ausleger_hinten", False),
        (step30_start_seconds + 0.8, "ts_ausleger_vorne", True),
        (step30_start_seconds + 1.1, "ts_ausleger_vorne", False),
        (step30_start_seconds + 1.9, "ts_ausleger_hinten", True),
    ])


def schedule_hrl_auslagern_conveyor_sequence(
    ls_inner_release_seconds: float = 9.2,
    ls_outer_detect_seconds: float = 15.0,
) -> List[Dict[str, Any]]:
    return schedule_ads_bool_sequence([
        (ls_inner_release_seconds, "ls_inner", False),
        (ls_outer_detect_seconds, "ls_outer", True),
    ])


def schedule_hrl_auslagern_encoder_sequence() -> List[Dict[str, Any]]:
    # Der SPS-Skill benoetigt nach dem Fix echte Encoderfortschritte.
    # Ohne diese Werte bleiben die Schritte 10/20/50/60 im Watchdog haengen.
    return schedule_ads_udint_sequence([
        (0.0, "horizontal_1", 0),
        (0.0, "horizontal_2", 0),
        (0.0, "vertical_1", 0),
        (0.0, "vertical_2", 0),
        (2.0, "horizontal_1", 1000),
        (2.0, "horizontal_2", 1000),
        (4.0, "vertical_1", 500),
        (4.0, "vertical_2", 500),
        (7.0, "horizontal_1", 0),
        (7.0, "horizontal_2", 0),
        (8.5, "vertical_1", 0),
        (8.5, "vertical_2", 0),
    ])


prepare_plc_for_skill_tests()
read_encoder_snapshot()

{'horizontal_1': 0, 'horizontal_2': 0, 'vertical_1': 0, 'vertical_2': 0}

In [132]:
# Beispiel: GVL_HRL_Sim.HRL_LS_aussen nach 10 Sekunden auf TRUE setzen.
# Du kannst entweder den Alias "ls_outer" oder den vollen Symbolnamen verwenden.
# Fuer die Diagnose kannst du danach read_hrl_signal_diagnostic() aufrufen.
#
# write_ads_bool_after_delay("ls_outer", True, 10.0)
# write_ads_bool_after_delay("GVL_HRL_Sim.HRL_LS_aussen", True, 10.0)

In [133]:
def build_client() -> Client:
    client = Client(OPC_ENDPOINT, timeout=OPC_REQUEST_TIMEOUT_SECONDS)
    client.application_uri = APPLICATION_URI
    client.secure_channel_timeout = OPC_SECURE_CHANNEL_TIMEOUT_MS
    client.session_timeout = OPC_SESSION_TIMEOUT_MS

    if USE_SECURITY:
        client.set_security_string(security_string)

    client.set_user(OPC_USERNAME)
    client.set_password(OPC_PASSWORD)
    return client


@contextmanager
def open_opcua():
    client = build_client()
    client.connect()
    try:
        yield client
    finally:
        client.disconnect()


def node_name(node: Any) -> str:
    browse_name = node.get_browse_name()
    return str(getattr(browse_name, "Name", browse_name))


def node_id_text(node: Any) -> str:
    return node.nodeid.to_string()


def node_class_name(node: Any) -> str:
    node_class = node.get_node_class()
    return str(getattr(node_class, "name", node_class))


def normalize_result(result: Any) -> Tuple[Any, ...]:
    if result is None:
        return tuple()
    if isinstance(result, tuple):
        return result
    if isinstance(result, list):
        return tuple(result)
    return (result,)


def is_method_node(node: Any) -> bool:
    return int(node.get_node_class()) == 4


def find_child_by_name(root_node: Any, expected_name: str) -> Any:
    for child in root_node.get_children():
        if node_name(child) == expected_name:
            return child
    raise LookupError(f"Child '{expected_name}' nicht unter '{node_name(root_node)}' gefunden.")


def find_descendant_by_name(root_node: Any, expected_name: str, max_depth: int = 10) -> Any:
    queue: List[Tuple[Any, int]] = [(root_node, 0)]
    while queue:
        node, depth = queue.pop(0)
        if node_name(node) == expected_name:
            return node
        if depth >= max_depth:
            continue
        for child in node.get_children():
            queue.append((child, depth + 1))
    raise LookupError(f"Node '{expected_name}' wurde im OPC-UA-Baum nicht gefunden.")


def iter_descendants(root_node: Any, max_depth: int = 10) -> Iterator[Tuple[Any, int]]:
    queue: List[Tuple[Any, int]] = [(root_node, 0)]
    while queue:
        node, depth = queue.pop(0)
        yield node, depth
        if depth >= max_depth:
            continue
        for child in node.get_children():
            queue.append((child, depth + 1))


def get_skillset_node(client: Client) -> Any:
    if SKILLSET_NODEID:
        return client.get_node(SKILLSET_NODEID)

    objects_node = client.get_objects_node()
    search_root = objects_node
    if PLC_ROOT_NAME:
        search_root = find_descendant_by_name(objects_node, PLC_ROOT_NAME, max_depth=10)
    return find_descendant_by_name(search_root, SKILLSET_NAME, max_depth=10)


def list_node_children(node: Any) -> List[Dict[str, str]]:
    return [
        {
            "name": node_name(child),
            "nodeid": node_id_text(child),
            "node_class": node_class_name(child),
        }
        for child in node.get_children()
    ]


def get_method_container_nodes(client: Client) -> Dict[str, Any]:
    skillset_node = get_skillset_node(client)
    result: Dict[str, Any] = {}
    for child, _depth in iter_descendants(skillset_node, max_depth=10):
        if child == skillset_node:
            continue
        method_children = [grandchild for grandchild in child.get_children() if is_method_node(grandchild)]
        if method_children:
            result[node_name(child)] = child
    return result


def get_container_method_nodes(container_node: Any) -> Dict[str, Any]:
    return {
        node_name(child): child
        for child in container_node.get_children()
        if is_method_node(child)
    }


def get_method_node(
    client: Client,
    *,
    container_name: Optional[str] = None,
    method_name: Optional[str] = None,
    method_nodeid: Optional[str] = None,
) -> Any:
    if method_nodeid:
        return client.get_node(method_nodeid)

    if not container_name or not method_name:
        raise ValueError("Bitte entweder method_nodeid oder container_name + method_name angeben.")

    container_nodes = get_method_container_nodes(client)
    if container_name not in container_nodes:
        raise LookupError(f"Methoden-Container '{container_name}' nicht gefunden.")

    method_nodes = get_container_method_nodes(container_nodes[container_name])
    if method_name not in method_nodes:
        raise LookupError(f"Methode '{method_name}' nicht unter '{container_name}' gefunden.")

    return method_nodes[method_name]


BUILTIN_NODEID_TO_VARIANT = {
    1: ua.VariantType.Boolean,
    2: ua.VariantType.SByte,
    3: ua.VariantType.Byte,
    4: ua.VariantType.Int16,
    5: ua.VariantType.UInt16,
    6: ua.VariantType.Int32,
    7: ua.VariantType.UInt32,
    8: ua.VariantType.Int64,
    9: ua.VariantType.UInt64,
    10: ua.VariantType.Float,
    11: ua.VariantType.Double,
    12: ua.VariantType.String,
    13: ua.VariantType.DateTime,
}


def get_argument_definitions(method_node: Any, property_name: str = "InputArguments") -> List[Dict[str, Any]]:
    try:
        property_node = method_node.get_child([f"0:{property_name}"])
    except Exception:
        return []

    definitions: List[Dict[str, Any]] = []
    for argument in property_node.get_value():
        data_type = getattr(argument, "DataType", None)
        data_type_identifier = getattr(data_type, "Identifier", None)
        definitions.append(
            {
                "name": getattr(argument, "Name", ""),
                "data_type": data_type,
                "data_type_nodeid": data_type.to_string() if data_type is not None else None,
                "variant_type": BUILTIN_NODEID_TO_VARIANT.get(int(data_type_identifier)) if data_type_identifier is not None else None,
            }
        )
    return definitions


def get_method_signature(method_node: Any) -> Dict[str, Any]:
    parent_node = method_node.get_parent()
    return {
        "method_name": node_name(method_node),
        "method_nodeid": node_id_text(method_node),
        "parent_name": node_name(parent_node),
        "parent_nodeid": node_id_text(parent_node),
        "inputs": [
            {
                "name": argument["name"],
                "data_type_nodeid": argument["data_type_nodeid"],
                "variant_type": str(argument["variant_type"]) if argument["variant_type"] is not None else None,
            }
            for argument in get_argument_definitions(method_node, "InputArguments")
        ],
        "outputs": [
            {
                "name": argument["name"],
                "data_type_nodeid": argument["data_type_nodeid"],
                "variant_type": str(argument["variant_type"]) if argument["variant_type"] is not None else None,
            }
            for argument in get_argument_definitions(method_node, "OutputArguments")
        ],
    }


def build_typed_method_arguments(method_node: Any, values_by_name: Dict[str, Any]) -> List[ua.Variant]:
    input_arguments = get_argument_definitions(method_node, "InputArguments")
    expected_names = [argument["name"] for argument in input_arguments]
    missing = [name for name in expected_names if name not in values_by_name]
    unexpected = sorted(set(values_by_name) - set(expected_names))

    if missing or unexpected:
        messages: List[str] = []
        if missing:
            messages.append(f"fehlend: {missing}")
        if unexpected:
            messages.append(f"unerwartet: {unexpected}")
        raise KeyError("Methodenargumente passen nicht zur Signatur (" + ", ".join(messages) + ").")

    typed_arguments: List[ua.Variant] = []
    for argument in input_arguments:
        value = values_by_name[argument["name"]]
        variant_type = argument["variant_type"]
        if variant_type is None:
            typed_arguments.append(ua.Variant(value))
        else:
            typed_arguments.append(ua.Variant(value, variant_type))
    return typed_arguments


def call_method_node(method_node: Any, values_by_name: Dict[str, Any]) -> Dict[str, Any]:
    parent_node = method_node.get_parent()
    input_arguments = get_argument_definitions(method_node, "InputArguments")
    output_arguments = get_argument_definitions(method_node, "OutputArguments")
    typed_arguments = build_typed_method_arguments(method_node, values_by_name)

    request = ua.CallMethodRequest()
    request.ObjectId = parent_node.nodeid
    request.MethodId = method_node.nodeid
    request.InputArguments = typed_arguments

    call_result = parent_node.server.call([request])[0]
    status_value = int(call_result.StatusCode.value)
    status_name, status_doc = ua.status_codes.get_name_and_doc(status_value)
    status_is_bad = bool(status_value & 0x80000000)
    status_is_uncertain = (status_value & 0xC0000000) == 0x40000000

    if status_is_bad:
        call_result.StatusCode.check()

    raw_outputs = tuple(variant.Value for variant in getattr(call_result, "OutputArguments", []))

    return {
        "method_name": node_name(method_node),
        "method_nodeid": node_id_text(method_node),
        "parent_name": node_name(parent_node),
        "parent_nodeid": node_id_text(parent_node),
        "status_code": status_value,
        "status_code_hex": f"0x{status_value:08X}",
        "status_name": status_name,
        "status_doc": status_doc,
        "status_is_good": call_result.StatusCode.is_good(),
        "status_is_uncertain": status_is_uncertain,
        "inputs": {argument["name"]: values_by_name[argument["name"]] for argument in input_arguments},
        "outputs": {
            argument["name"]: raw_outputs[index] if index < len(raw_outputs) else None
            for index, argument in enumerate(output_arguments)
        },
        "raw_outputs": raw_outputs,
    }


def call_method_by_nodeid(client: Client, method_nodeid: str, values_by_name: Dict[str, Any]) -> Dict[str, Any]:
    return call_method_node(get_method_node(client, method_nodeid=method_nodeid), values_by_name)


def get_hrl_skill_nodes(client: Client) -> Dict[str, Any]:
    skillset_node = get_skillset_node(client)
    result: Dict[str, Any] = {}
    for child, _depth in iter_descendants(skillset_node, max_depth=10):
        if child == skillset_node:
            continue
        method_names = {node_name(grandchild) for grandchild in child.get_children()}
        if {"Start", "CheckState", "Abort"}.issubset(method_names):
            result[node_name(child)] = child
    return result


def get_hrl_skill_node(client: Client, skill_name: str) -> Any:
    skill_nodes = get_hrl_skill_nodes(client)
    if skill_name not in skill_nodes:
        raise LookupError(f"Skill '{skill_name}' wurde nicht gefunden. Verfuegbar: {sorted(skill_nodes)}")
    return skill_nodes[skill_name]


def get_skill_method_node(skill_node: Any, method_name: str) -> Any:
    return find_child_by_name(skill_node, method_name)


def call_skill_method_node(skill_node: Any, method_name: str, values_by_name: Dict[str, Any]) -> Dict[str, Any]:
    return call_method_node(get_skill_method_node(skill_node, method_name), values_by_name)


def list_server_endpoints() -> List[Dict[str, str]]:
    probe = Client(OPC_ENDPOINT)
    endpoints = probe.connect_and_get_server_endpoints()
    rows = []
    for ep in endpoints:
        rows.append({
            "EndpointUrl": ep.EndpointUrl,
            "SecurityMode": str(ep.SecurityMode),
            "SecurityPolicyUri": ep.SecurityPolicyUri,
        })
    return rows


def list_hrl_methods() -> Dict[str, List[Dict[str, str]]]:
    with open_opcua() as client:
        container_nodes = get_method_container_nodes(client)
        return {
            container_name: [
                {
                    "method_name": node_name(method_node),
                    "method_nodeid": node_id_text(method_node),
                }
                for method_node in get_container_method_nodes(container_node).values()
            ]
            for container_name, container_node in container_nodes.items()
        }

In [134]:
endpoints = list_server_endpoints()
endpoints

[{'EndpointUrl': 'opc.tcp://DESKTOP-LNJR8E0:4840',
  'SecurityMode': '3',
  'SecurityPolicyUri': 'http://opcfoundation.org/UA/SecurityPolicy#Basic256Sha256'},
 {'EndpointUrl': 'opc.tcp://DESKTOP-LNJR8E0:4840',
  'SecurityMode': '3',
  'SecurityPolicyUri': 'http://opcfoundation.org/UA/SecurityPolicy#Aes256_Sha256_RsaPss'},
 {'EndpointUrl': 'opc.tcp://DESKTOP-LNJR8E0:4840',
  'SecurityMode': '3',
  'SecurityPolicyUri': 'http://opcfoundation.org/UA/SecurityPolicy#Aes128_Sha256_RsaOaep'}]

In [135]:
with open_opcua() as client:
    root_name = node_name(client.get_root_node())
    objects_name = node_name(client.get_objects_node())
    skillset_node = get_skillset_node(client)
    connection_info = {
        "connect_ok": True,
        "username": OPC_USERNAME,
        "use_security": USE_SECURITY,
        "root": root_name,
        "objects": objects_name,
        "skillset": node_name(skillset_node),
        "skillset_nodeid": node_id_text(skillset_node),
    }

print(connection_info)

{'connect_ok': True, 'username': 'AdminVD', 'use_security': True, 'root': 'Root', 'objects': 'Objects', 'skillset': 'HRL_SkillSet', 'skillset_nodeid': 'ns=4;s=HRL_SkillSet'}


In [136]:
skill_methods = list_hrl_methods()
skill_methods

{'HRL_NMethod_Auslagern': [{'method_name': 'HRL_NMethod_Auslagern',
   'method_nodeid': 'ns=4;s=HRL_SkillSet.HRL_NMethod_Auslagern.HRL_NMethod_Auslagern'}],
 'HRL_Skill_CB_MoveForwards': [{'method_name': 'JobMethode_HorizontalMove',
   'method_nodeid': 'ns=4;s=HRL_SkillSet.HRL_Skill_CB_MoveForwards.JobMethode_HorizontalMove'}],
 'HRL_Skill_CB_MoveBackwards': [{'method_name': 'JobMethode_HorizontalMove',
   'method_nodeid': 'ns=4;s=HRL_SkillSet.HRL_Skill_CB_MoveBackwards.JobMethode_HorizontalMove'}],
 'HRL_Skill_RGB_MoveBackwards': [{'method_name': 'JobMethode_HorizontalMove',
   'method_nodeid': 'ns=4;s=HRL_SkillSet.HRL_Skill_RGB_MoveBackwards.JobMethode_HorizontalMove'}],
 'HRL_Skill_RGB_MoveForwards': [{'method_name': 'JobMethode_HorizontalMove',
   'method_nodeid': 'ns=4;s=HRL_SkillSet.HRL_Skill_RGB_MoveForwards.JobMethode_HorizontalMove'}],
 'HRL_Skill_RGB_MoveVerticalForwards': [{'method_name': 'JobMethode_VerticalMove',
   'method_nodeid': 'ns=4;s=HRL_SkillSet.HRL_Skill_RGB_MoveV

In [137]:
auslagern_start_inputs = {
    "MethodCall": True,
    "HorizontalShelf_I1": 1000,
    "HorizontalShelf_I2": 1000,
    "VerticalShelf_I1": 500,
    "VerticalShelf_I2": 500,
    "HorizontalTransfer_I1": 0,
    "HorizontalTransfer_I2": 0,
    "VerticalTransfer_I1": 0,
    "VerticalTransfer_I2": 0,
    "Timeout": 30000,
}


def get_hrl_auslagern_container(client: Client) -> Any:
    container_nodes = get_method_container_nodes(client)
    if HRL_AUSLAGERN_SKILL_NAME not in container_nodes:
        raise LookupError(
            f"Methoden-Container '{HRL_AUSLAGERN_SKILL_NAME}' wurde nicht gefunden. "
            f"Verfuegbar: {sorted(container_nodes)}"
        )
    return container_nodes[HRL_AUSLAGERN_SKILL_NAME]


def get_hrl_auslagern_method_node(client: Client) -> Any:
    container_node = get_hrl_auslagern_container(client)
    return find_child_by_name(container_node, HRL_AUSLAGERN_SKILL_NAME)


def get_hrl_auslagern_signatures() -> Dict[str, Any]:
    with open_opcua() as client:
        container_node = get_hrl_auslagern_container(client)
        method_nodes = get_container_method_nodes(container_node)
        method_node = get_hrl_auslagern_method_node(client)
        return {
            "container_name": node_name(container_node),
            "container_nodeid": node_id_text(container_node),
            "available_methods": {
                name: node_id_text(node)
                for name, node in method_nodes.items()
            },
            "job_method": get_method_signature(method_node),
        }


def call_hrl_auslagern(client: Client, start_values: Dict[str, Any]) -> Dict[str, Any]:
    return call_method_node(get_hrl_auslagern_method_node(client), start_values)


def start_hrl_auslagern(client: Client, start_values: Dict[str, Any]) -> Dict[str, Any]:
    return call_hrl_auslagern(client, start_values)


def check_hrl_auslagern(client: Client, handle: int) -> Dict[str, Any]:
    raise NotImplementedError(
        "Die aktuelle OPC-UA-Struktur exportiert fuer HRL_NMethod_Auslagern nur "
        "JobMethode_HRL_Auslagern, aber kein separates CheckState()."
    )


def abort_hrl_auslagern(client: Client, handle: int) -> Dict[str, Any]:
    raise NotImplementedError(
        "Die aktuelle OPC-UA-Struktur exportiert fuer HRL_NMethod_Auslagern nur "
        "JobMethode_HRL_Auslagern, aber kein separates Abort()."
    )


def run_hrl_auslagern(
    start_values: Dict[str, Any],
    *,
    ls_outer_after_seconds: Optional[float] = None,
    ls_outer_value: bool = True,
) -> Dict[str, Any]:
    prepared = prepare_plc_for_skill_tests()
    scheduled = None
    if ls_outer_after_seconds is not None:
        scheduled = schedule_ads_bool_write("ls_outer", ls_outer_value, ls_outer_after_seconds)
    with open_opcua() as client:
        call_result = call_hrl_auslagern(client, start_values)

    return {
        "prepared": prepared,
        "scheduled_ls_outer": scheduled,
        "call": call_result,
    }


def run_hrl_auslagern_until_finish(
    start_values: Dict[str, Any],
    *,
    ls_outer_after_seconds: Optional[float] = None,
    ls_outer_value: bool = True,
    wait_timeout_seconds: float = 60.0,
) -> Dict[str, Any]:
    return run_hrl_auslagern(
        start_values,
        ls_outer_after_seconds=ls_outer_after_seconds,
        ls_outer_value=ls_outer_value,
    )


def run_and_abort_hrl_auslagern(
    start_values: Dict[str, Any],
    *,
    abort_after_seconds: float = ABORT_DELAY,
) -> Dict[str, Any]:
    raise NotImplementedError(
        "run_and_abort_hrl_auslagern() setzt ein PLCopen-Interface mit Start/CheckState/Abort voraus. "
        "Aktuell ist nur JobMethode_HRL_Auslagern exportiert."
    )


In [138]:
# Direkter Methodenaufruf ueber JobMethode_HRL_Auslagern.
# Fuer den aktuellen SPS-Stand gilt TRUE = Detektion.
# ls_inner wird in der Grundinitialisierung bereits auf TRUE gesetzt.
# ls_outer muss waehrend des laufenden Methodenaufrufs auf TRUE wechseln,
# damit Schritt 70 sauber in den Erfolgszustand uebergeht.
# Der Rueckgabedatensatz enthaelt bei Bedarf status_code_hex, status_name und status_doc.
# Fuer den gepatchten SPS-Stand werden auch Encoder-, Ausleger- und Foerderbandverlaeufe simuliert.
# schedule_hrl_auslagern_encoder_sequence()
# schedule_hrl_auslagern_ausleger_sequence()
# schedule_hrl_auslagern_conveyor_sequence()
# normal_result = run_hrl_auslagern(auslagern_start_inputs)
# normal_result
prepare_plc_for_skill_tests()
schedule_hrl_auslagern_encoder_sequence()
schedule_hrl_auslagern_ausleger_sequence()
schedule_hrl_auslagern_conveyor_sequence()
direct_result = run_hrl_auslagern(auslagern_start_inputs)
direct_result

{'prepared': {'sim': True,
  'notaus_a': True,
  'notaus_b': True,
  'hrl_reset_button': True,
  'hrl_start_button': True,
  'hrl_confirmation_button': False,
  'vsg_reset_button': True,
  'vsg_start_button': True,
  'vsg_confirmation_button': False,
  'ts_horizontal': False,
  'ts_vertical': False,
  'ts_ausleger_vorne': False,
  'ts_ausleger_hinten': True,
  'ls_inner': True,
  'ls_outer': False},
 'scheduled_ls_outer': None,
 'call': {'method_name': 'HRL_NMethod_Auslagern',
  'method_nodeid': 'ns=4;s=HRL_SkillSet.HRL_NMethod_Auslagern.HRL_NMethod_Auslagern',
  'parent_name': 'HRL_NMethod_Auslagern',
  'parent_nodeid': 'ns=4;s=HRL_SkillSet.HRL_NMethod_Auslagern',
  'status_code': 1073741824,
  'status_code_hex': '0x40000000',
  'status_name': 'Uncertain',
  'status_doc': 'The operation completed however its outputs may not be usable.',
  'status_is_good': False,
  'status_is_uncertain': True,
  'inputs': {'MethodCall': True,
   'HorizontalShelf_I1': 1000,
   'HorizontalShelf_I2': 100

In [122]:
write_ads_bool("ls_outer", False)
write_ads_bool("OPCUA.Diag_finished", False)

deadline = time.time() + 1.0
while time.time() < deadline:
    if read_ads_bool("GVL_HRL.HRL_LS_aussen")["value"] is False:
        break
    time.sleep(0.01)

time.sleep(0.05)  # noch 1-2 SPS-Zyklen Puffer

{
    "ls_outer_sim": read_ads_bool("ls_outer"),
    "ls_outer_runtime": read_ads_bool("GVL_HRL.HRL_LS_aussen"),
    "f1": read_ads_bool("VSG_SkillSet.VSG_OperatingModes.F1"),
    "d2": read_ads_bool("VSG_SkillSet.VSG_OperatingModes.D2"),
}


{'ls_outer_sim': {'symbol': 'GVL_HRL_Sim.HRL_LS_aussen', 'value': False},
 'ls_outer_runtime': {'symbol': 'GVL_HRL.HRL_LS_aussen', 'value': False},
 'f1': {'symbol': 'VSG_SkillSet.VSG_OperatingModes.F1', 'value': False},
 'd2': {'symbol': 'VSG_SkillSet.VSG_OperatingModes.D2', 'value': True}}

In [125]:
VSG_COMPRESSOR_METHOD_NODEID = "ns=4;s=VSG_SkillSet.VSG_Skill_SuctionProcess.JobMethode_CompressorControl"

vsg_ok_inputs = {
    "MethodCall": True,
    "DestinationReached": True,
    "EncoderTargetPosition_01": 100,
    "EncoderTargetPosition_02": 100,
    "EncoderTargetPosition_03": 200,
    "EncoderTargetPosition_04": 200,
    "EncoderTargetPosition_05": 300,
    "EncoderTargetPosition_06": 300,
}

write_ads_bool("sim", True)
write_ads_bool("notaus_a", True)
write_ads_bool("notaus_b", True)
write_ads_bool("vsg_reset_button", True)
write_ads_bool("vsg_start_button", True)
write_ads_bool("vsg_confirmation_button", False)
#write_ads_bool("ls_outer", True)
time.sleep(0.1)

schedule_ads_udint_sequence([
    (0.0, "GVL_VSG_Sim.VSG_Enc_drehen_I1", 0),
    (0.0, "GVL_VSG_Sim.VSG_Enc_drehen_I2", 0),
    (0.0, "GVL_VSG_Sim.VSG_Enc_horizontal_I1", 0),
    (0.0, "GVL_VSG_Sim.VSG_Enc_horizontal_I2", 0),
    (0.0, "GVL_VSG_Sim.VSG_Enc_vertikal_I1", 0),
    (0.0, "GVL_VSG_Sim.VSG_Enc_vertikal_I2", 0),

    (0.8, "GVL_VSG_Sim.VSG_Enc_drehen_I1", 100),
    (0.8, "GVL_VSG_Sim.VSG_Enc_drehen_I2", 100),
    (0.8, "GVL_VSG_Sim.VSG_Enc_horizontal_I1", 200),
    (0.8, "GVL_VSG_Sim.VSG_Enc_horizontal_I2", 200),
    (0.8, "GVL_VSG_Sim.VSG_Enc_vertikal_I1", 300),
    (0.8, "GVL_VSG_Sim.VSG_Enc_vertikal_I2", 300),
])

with open_opcua() as client:
    vsg_ok_result = call_method_by_nodeid(
        client,
        VSG_COMPRESSOR_METHOD_NODEID,
        vsg_ok_inputs,
    )

vsg_ok_result

{'method_name': 'JobMethode_CompressorControl',
 'method_nodeid': 'ns=4;s=VSG_SkillSet.VSG_Skill_SuctionProcess.JobMethode_CompressorControl',
 'parent_name': 'VSG_Skill_SuctionProcess',
 'parent_nodeid': 'ns=4;s=VSG_SkillSet.VSG_Skill_SuctionProcess',
 'status_code': 1073741824,
 'status_code_hex': '0x40000000',
 'status_name': 'Uncertain',
 'status_doc': 'The operation completed however its outputs may not be usable.',
 'status_is_good': False,
 'status_is_uncertain': True,
 'inputs': {'MethodCall': True,
  'DestinationReached': True,
  'EncoderTargetPosition_01': 100,
  'EncoderTargetPosition_02': 100,
  'EncoderTargetPosition_03': 200,
  'EncoderTargetPosition_04': 200,
  'EncoderTargetPosition_05': 300,
  'EncoderTargetPosition_06': 300},
 'outputs': {'State': 0, 'IsDone': True, 'HasError': True, 'ErrorCode': 1001},
 'raw_outputs': (0, True, True, 1001)}

In [126]:
time.sleep(0.2)
{
    "d2": read_ads_bool("VSG_SkillSet.VSG_OperatingModes.D2"),
    "f1": read_ads_bool("VSG_SkillSet.VSG_OperatingModes.F1"),
    "err": read_ads_bool("VSG_SkillSet.VSG_ErrorDetected"),
}

{'d2': {'symbol': 'VSG_SkillSet.VSG_OperatingModes.D2', 'value': True},
 'f1': {'symbol': 'VSG_SkillSet.VSG_OperatingModes.F1', 'value': False},
 'err': {'symbol': 'VSG_SkillSet.VSG_ErrorDetected', 'value': True}}

In [1]:
write_ads_bool("VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected", False)
write_ads_bool("OPCUA.Diag_finished", True)

{
    "vsg_error": read_ads_bool("VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected"),
    "diag_finished": read_ads_bool("OPCUA.Diag_finished"),
}

NameError: name 'write_ads_bool' is not defined